# JFDS Experiment — 04: Significance, Robustness & Adaptive-λ

**Part of a 5-notebook series.** This is the notebook that carries the paper's confirmatory
claims: paired significance tests (Wilcoxon + Holm–Bonferroni correction across all comparisons),
bootstrap confidence intervals, group-fairness breakdown by user demographics, aggregate-diversity
/ exposure-equity analysis (Lorenz curves + chi-square), the Adaptive-λ JFDS variant, and a
30-independent-resample robustness check.

Contents:
- **R7 / R7b** — Per-user paired significance tests (Wilcoxon), Holm–Bonferroni corrected
- **R8** — Bootstrap confidence intervals on the effect size
- **R9** — Group fairness across gender / age / occupation
- **R10 / R10b** — Aggregate diversity, exposure Lorenz curves, bootstrap CI on aggregate diversity
- **Adaptive-λ JFDS** — letting λ_f, λ_d grow with list position instead of staying fixed
- **30-resample robustness check** — is the JFDS-vs-baseline ranking stable across independent
  user subsamples, or an artifact of one particular sample
- **Conclusion** — hypothesis-test summary, limitations, practical implications, future work

## Setup — load artifacts from `02_main_experiments.ipynb`

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
from IPython.display import display, Markdown

with open('jfds_artifacts.pkl', 'rb') as f:
    ARTIFACTS = pickle.load(f)
globals().update(ARTIFACTS)

plt.rcParams['figure.dpi'] = 110
print(f"Loaded {len(ARTIFACTS)} artifacts from jfds_artifacts.pkl")
print(f"N_USERS={N_USERS}  N_ITEMS={N_ITEMS}  K={K}  POOL_SIZE={POOL_SIZE}")
print("best_lambdas:", best_lambdas)


In [ ]:
# -- Re-declare the core re-ranking + metric functions --
# These are pure functions of (tier, TARGET_SHARE, GENRE_SIM, genre_vec, pop_norm),
# all of which were just loaded from jfds_artifacts.pkl -- no retraining needed.
# Kept byte-identical to their definitions in 02_main_experiments.ipynb so results
# are guaranteed reproducible across notebooks.

def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map  = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])

def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term_fast(cand_idx, selected_idx):
    """Uses pre-computed GENRE_SIM -- no per-call cosine computation."""
    if not selected_idx:
        return 1.0
    return float(1.0 - GENRE_SIM[cand_idx, selected_idx].mean())

def jfds_score(cand_idx, rel_value, selected, tier_counts, step, lam_f, lam_d):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f, lam_d, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)

def mmr_max_sim(cand_idx, selected_idx):
    if not selected_idx:
        return 0.0
    return float(GENRE_SIM[cand_idx, selected_idx].max())

def mmr_score(cand_idx, rel_value, selected, tier_counts, step, lam=MMR_LAMBDA):
    return lam * rel_value - (1 - lam) * mmr_max_sim(cand_idx, selected)

def mmr_rerank(cand_ids, cand_rel, lam=MMR_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, mmr_score, k=k, lam=lam)

def fairness_only_score(cand_idx, rel_value, selected, tier_counts, step, lam_f=FAIR_ONLY_LAMBDA):
    return (1 - lam_f) * rel_value + lam_f * fair_boost(cand_idx, tier_counts, step)

def fairness_only_rerank(cand_ids, cand_rel, lam_f=FAIR_ONLY_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, fairness_only_score, k=k, lam_f=lam_f)

def ild(rec_list):
    """Intra-list diversity: 1 - mean pairwise genre-cosine similarity."""
    if len(rec_list) < 2:
        return 0.0
    idx = np.array(rec_list)
    sub = GENRE_SIM[np.ix_(idx, idx)]
    n = len(idx)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(1.0 - sub[mask].mean())

def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec   = rec_list[:k]
    hits  = len(set(rec) & relevant_set)
    prec  = hits / k
    rec_r = hits / max(1, len(relevant_set))
    dcg   = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    idcg  = sum(g / np.log2(r + 2) for r, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return prec, rec_r, (dcg / idcg if idcg > 0 else 0.0)

def novelty_fairness(rec_list):
    """Mean (1 - normalised popularity): higher = more novel/niche items recommended."""
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))

def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

def aggregate_diversity(rec_lists_dict):
    all_items = set()
    for rec in rec_lists_dict.values():
        all_items.update(rec)
    return len(all_items)

def exposure_vector(rec_lists_dict, n_items=N_ITEMS):
    exp = np.zeros(n_items)
    for rec in rec_lists_dict.values():
        for i in rec:
            exp[i] += 1
    return exp

print('Core re-ranking + metric functions re-declared (greedy_rerank, topk_rerank, jfds_*, '
      'mmr_*, fairness_only_*, ild, precision_recall_ndcg, novelty_fairness, shannon_entropy, '
      'gini, calibration_kl, aggregate_diversity, exposure_vector)')


In [ ]:
# -- Adaptive-lambda schedule (same definition as in 02's "v3 Additions" section) --
DEFAULT_SCHEDULE = dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0)

def schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p=1.0):
    """t_k = step / (k_total - 1); lambda grows from *_start to *_end as t_k^p."""
    t = step / max(1, k_total - 1)
    lam_f = lf_start + (lf_end - lf_start) * (t ** p)
    lam_d = ld_start + (ld_end - ld_start) * (t ** p)
    return lam_f, lam_d

def adaptive_jfds_score(cand_idx, rel_value, selected, tier_counts, step, k_total=K,
                         lf_start=DEFAULT_SCHEDULE['lf_start'], lf_end=DEFAULT_SCHEDULE['lf_end'],
                         ld_start=DEFAULT_SCHEDULE['ld_start'], ld_end=DEFAULT_SCHEDULE['ld_end'],
                         p=DEFAULT_SCHEDULE['p']):
    lam_f, lam_d = schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p)
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def adaptive_jfds_rerank(cand_ids, cand_rel, k=K, **schedule_params):
    params = {**DEFAULT_SCHEDULE, **schedule_params}
    return greedy_rerank(cand_ids, cand_rel, adaptive_jfds_score, k=k, k_total=k, **params)

print('Adaptive-lambda JFDS equation re-declared - default schedule:', DEFAULT_SCHEDULE)


---
## R7 — Paired Significance Tests

In [ ]:
sig_rows = []
BASELINE_METHODS = ['TopK', 'MMR', 'FairOnly']

for model_name in ['SVD', 'BPR']:
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')

    for baseline_method in BASELINE_METHODS:
        base_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == baseline_method)].set_index('user')
        common  = base_df.index.intersection(jfds_df.index)
        base_c, jfds_c = base_df.loc[common], jfds_df.loc[common]

        for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
            a, b = base_c[metric].values, jfds_c[metric].values
            diff = b - a
            try:
                w_stat, w_p = stats.wilcoxon(a, b)
            except ValueError:
                w_stat, w_p = np.nan, np.nan
            t_stat, t_p = stats.ttest_rel(b, a)
            sig_rows.append({
                'base_model': model_name, 'baseline': baseline_method, 'metric': metric,
                'mean_baseline': a.mean(), 'mean_JFDS': b.mean(),
                'mean_delta': diff.mean(), 'wilcoxon_p': w_p, 'ttest_p': t_p,
                'JFDS_wins': int(np.sum(diff > 1e-9)),
                'JFDS_losses': int(np.sum(diff < -1e-9)),
                'ties': len(diff) - int(np.sum(diff > 1e-9)) - int(np.sum(diff < -1e-9)),
                'n_users': len(common),
            })

sig_table = pd.DataFrame(sig_rows).round(5)
sig_table


---
## R7b — Holm–Bonferroni Correction

`sig_table` above now holds `36` separate per-user paired tests (3 baselines
× 6 metrics × 2 base models). Reporting raw p-values from that many
comparisons without correction overstates significance — a Holm–Bonferroni
correction controls the family-wise error rate across all of them at once.


In [ ]:
# ── Holm–Bonferroni correction across all per-user paired tests ──────────
try:
    from statsmodels.stats.multitest import multipletests
    HAVE_STATSMODELS = True
except ImportError:
    HAVE_STATSMODELS = False
    print("statsmodels not available — falling back to a manual Holm–Bonferroni implementation.")

def holm_bonferroni(pvals, alpha=0.05):
    """Manual fallback: returns (reject_bool_array, corrected_pvals) in original order."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    corrected = np.empty(n)
    running_max = 0.0
    for rank, i in enumerate(order):
        adj = (n - rank) * pvals[i]
        running_max = max(running_max, adj)
        corrected[i] = min(running_max, 1.0)
    reject = corrected < alpha
    return reject, corrected

ALPHA = 0.05
pvals = sig_table['wilcoxon_p'].fillna(1.0).values

if HAVE_STATSMODELS:
    reject, p_corrected, _, _ = multipletests(pvals, alpha=ALPHA, method='holm')
else:
    reject, p_corrected = holm_bonferroni(pvals, alpha=ALPHA)

sig_table['wilcoxon_p_holm'] = p_corrected
sig_table['significant_holm'] = reject

print('=== Per-user paired tests: JFDS vs each baseline, AFTER Holm–Bonferroni correction ===')
display_cols = ['base_model', 'baseline', 'metric', 'mean_delta', 'wilcoxon_p',
                 'wilcoxon_p_holm', 'significant_holm']
print(sig_table[display_cols].to_string(index=False))

n_sig_holm = int(sig_table['significant_holm'].sum())
print(f"\n{n_sig_holm} / {len(sig_table)} comparisons remain significant after Holm correction (α={ALPHA}).")


---
## R8 — Bootstrap Confidence Intervals

In [ ]:
def bootstrap_delta_ci(a, b, n_boot=2000, seed=RANDOM_SEED):
    rng   = np.random.default_rng(seed)
    diffs = b - a
    boots = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(n_boot)])
    return diffs.mean(), np.percentile(boots, [2.5, 97.5])

boot_rows = []
for model_name in ['SVD', 'BPR']:
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')
    for baseline_method in BASELINE_METHODS:
        base_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == baseline_method)].set_index('user')
        common  = base_df.index.intersection(jfds_df.index)
        base_c, jfds_c = base_df.loc[common], jfds_df.loc[common]
        for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
            mean_delta, ci = bootstrap_delta_ci(base_c[metric].values, jfds_c[metric].values)
            boot_rows.append({'base_model': model_name, 'baseline': baseline_method, 'metric': metric,
                              'mean_delta': mean_delta, 'ci_low': ci[0], 'ci_high': ci[1]})

boot_df = pd.DataFrame(boot_rows)
display(boot_df.round(5))

fig, axes = plt.subplots(len(BASELINE_METHODS), 2, figsize=(13, 14), sharey=False)
for row, baseline_method in enumerate(BASELINE_METHODS):
    for col, model_name in enumerate(['SVD', 'BPR']):
        ax  = axes[row][col]
        sub = boot_df[(boot_df.base_model == model_name) & (boot_df.baseline == baseline_method)].reset_index(drop=True)
        y   = np.arange(len(sub))
        colors_b = [C_JFDS if (lo > 0 or hi < 0) else 'grey' for lo, hi in zip(sub.ci_low, sub.ci_high)]

        # ecolor doesn't accept a per-point color array, so draw each errorbar individually
        for yi, md, lo, hi, col_ in zip(y, sub['mean_delta'], sub['ci_low'], sub['ci_high'], colors_b):
            ax.errorbar(md, yi,
                        xerr=[[md - lo], [hi - md]],
                        fmt='o', color='black', ecolor=col_, elinewidth=3, capsize=4)

        ax.axvline(0, color='grey', ls='--', lw=1)
        ax.set_yticks(y); ax.set_yticklabels(sub['metric'])
        ax.set_xlabel('mean(JFDS − baseline)  with 95% bootstrap CI')
        ax.set_title(f'{model_name} — JFDS vs {baseline_method}', fontsize=10)

plt.suptitle('R8 — Effect Size with Uncertainty vs. Every Baseline (colored = CI excludes zero)', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('r8_bootstrap_ci.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R9 — Group Fairness by User Demographics  ✨ NEW

Break down NDCG, F (novelty), and D (ILD) by Gender, Age bucket, and Occupation.
Answers: *does JFDS improve recommendations equally across all user groups, or does it help some while hurting others?*


In [ ]:
# Merge user demographics into metrics_long
demo_cols = ['u_idx', 'Gender', 'Age', 'Occupation']
metrics_demo = metrics_long.merge(
    users_df[demo_cols].rename(columns={'u_idx': 'user'}),
    on='user', how='left'
)

# Age bucketing
age_map = {1: '<18', 18: '18-24', 25: '25-34', 35: '35-44', 45: '45-49', 50: '50-55', 56: '56+'}
metrics_demo['Age_label'] = metrics_demo['Age'].map(age_map).fillna('Unknown')

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
group_cols  = ['Gender', 'Age_label', 'Occupation']
group_titles = ['Gender', 'Age group', 'Occupation']
metric_pairs = [('ndcg', 'NDCG@10'), ('F', 'Fairness (novelty)'), ('D', 'Diversity (ILD)')]

for col_idx, (demo_col, demo_title) in enumerate(zip(group_cols, group_titles)):
    for row_idx, (metric, metric_label) in enumerate(metric_pairs):
        ax = axes[row_idx][col_idx]
        grp = (metrics_demo.groupby([demo_col, 'method', 'base_model'])[metric]
               .mean().reset_index())

        groups = sorted(grp[demo_col].unique())
        x      = np.arange(len(groups))
        width  = 0.2
        offsets = {'TopK_SVD': -1.5, 'JFDS_SVD': -0.5, 'TopK_BPR': 0.5, 'JFDS_BPR': 1.5}
        colors  = {'TopK_SVD': C_TOPK, 'JFDS_SVD': C_JFDS,
                   'TopK_BPR': '#85C1E9', 'JFDS_BPR': '#F1948A'}

        for method in ['TopK', 'JFDS']:
            for bm in ['SVD', 'BPR']:
                sub   = grp[(grp['method'] == method) & (grp['base_model'] == bm)]
                vals  = [sub[sub[demo_col] == g][metric].values[0]
                         if len(sub[sub[demo_col] == g]) > 0 else 0 for g in groups]
                key   = f'{method}_{bm}'
                ax.bar(x + offsets[key] * width, vals, width,
                       label=key, color=colors[key], alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=8)
        ax.set_ylabel(metric_label, fontsize=9)
        ax.set_title(f'{metric_label} by {demo_title}', fontsize=9)
        if row_idx == 0 and col_idx == 2:
            ax.legend(fontsize=7, loc='upper right')

plt.suptitle('R9 — Group Fairness: JFDS vs Top-K across User Demographics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('r9_group_fairness.png', dpi=110, bbox_inches='tight')
plt.show()

# Statistical test: does JFDS help all groups equally? (Kruskal-Wallis on delta per demo group)
print("\nKruskal-Wallis test: is the JFDS gain in NDCG uniform across groups?")
for demo_col, demo_title in zip(group_cols, group_titles):
    for bm in ['SVD', 'BPR']:
        topk_d = metrics_demo[(metrics_demo.method=='TopK') & (metrics_demo.base_model==bm)].set_index('user')
        jfds_d = metrics_demo[(metrics_demo.method=='JFDS') & (metrics_demo.base_model==bm)].set_index('user')
        common = topk_d.index.intersection(jfds_d.index)
        delta_df = pd.DataFrame({
            'delta_ndcg': jfds_d.loc[common, 'ndcg'].values - topk_d.loc[common, 'ndcg'].values,
            demo_col: jfds_d.loc[common, demo_col].values,
        })
        groups_data = [g['delta_ndcg'].values for _, g in delta_df.groupby(demo_col) if len(g) > 5]
        if len(groups_data) >= 2:
            h_stat, p_val = stats.kruskal(*groups_data)
            print(f"  {bm} × {demo_title}: H={h_stat:.2f}  p={p_val:.4f}"
                  f"  {'← unequal gains' if p_val < 0.05 else '(uniform gains)'}")


---
## R10 — Aggregate Diversity & Exposure Lorenz Curve  ✨ NEW

Answers:
1. How many *distinct* items does each strategy recommend across all users? (Aggregate diversity)
2. How equitably is exposure distributed across items? (Lorenz curve + Gini)
3. Is exposure significantly different between Top-K and JFDS for head/mid/tail items? (Chi-square)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

agg_stats = []
for (model_name, method), rec_lists in lists.items():
    exp    = exposure_vector(rec_lists)
    agg_d  = aggregate_diversity(rec_lists)
    cov    = (exp > 0).sum() / N_ITEMS

    # Lorenz curve
    exp_sorted = np.sort(exp)
    cum_exp    = np.cumsum(exp_sorted) / exp_sorted.sum()
    cum_items  = np.linspace(0, 1, len(exp_sorted))
    exp_gini   = gini(exp)

    ax_idx = (0 if model_name == 'SVD' else 1, 0 if method == 'TopK' else 1)
    ax     = axes[ax_idx[0]][ax_idx[1]]
    ax.plot(cum_items, cum_exp, lw=2,
            color=C_TOPK if method == 'TopK' else C_JFDS,
            label=f'{model_name}-{method} (Gini={exp_gini:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Perfect equality')
    ax.fill_between(cum_items, cum_items, cum_exp, alpha=0.12,
                    color=C_TOPK if method == 'TopK' else C_JFDS)
    ax.set_xlabel('Fraction of items (sorted by exposure)')
    ax.set_ylabel('Fraction of total exposure')
    ax.set_title(f'{model_name} – {method}\nAggregate diversity: {agg_d:,} items  |  Coverage: {cov:.2%}',
                 fontsize=10)
    ax.legend(fontsize=9)

    # Tier-level exposure shares
    tier_exp = {t: exp[tier == t].sum() for t in TIERS}
    total_exp = sum(tier_exp.values())
    tier_shares = {t: tier_exp[t] / total_exp for t in TIERS}
    agg_stats.append({
        'base_model': model_name, 'method': method,
        'agg_diversity': agg_d, 'coverage': round(cov, 4), 'exposure_gini': round(exp_gini, 4),
        **{f'{t}_share': round(tier_shares[t], 4) for t in TIERS},
    })

plt.suptitle('R10 — Lorenz Curves of Item Exposure (area below diagonal = inequality)', fontsize=12)
plt.tight_layout()
plt.savefig('r10_lorenz_curves.png', dpi=110, bbox_inches='tight')
plt.show()

agg_df = pd.DataFrame(agg_stats).set_index(['base_model', 'method'])
display(agg_df)

# Chi-square test: are tier exposure shares significantly different JFDS vs TopK?
print("\nChi-square test: JFDS vs Top-K tier exposure distribution")
for bm in ['SVD', 'BPR']:
    exp_topk = exposure_vector(lists[(bm, 'TopK')])
    exp_jfds = exposure_vector(lists[(bm, 'JFDS')])

    obs_full = np.array([[exp_topk[tier == t].sum() for t in TIERS],
                          [exp_jfds[tier == t].sum()  for t in TIERS]])

    # chi2_contingency requires every column total > 0 (else expected freq = 0).
    # Drop tiers that got zero exposure from BOTH methods before testing.
    col_totals   = obs_full.sum(axis=0)
    keep_mask    = col_totals > 0
    dropped_tiers = [t for t, keep in zip(TIERS, keep_mask) if not keep]
    kept_tiers    = [t for t, keep in zip(TIERS, keep_mask) if keep]
    obs = obs_full[:, keep_mask]

    if dropped_tiers:
        print(f"  {bm}: dropping tier(s) with zero total exposure from the test: {dropped_tiers}")

    if obs.shape[1] < 2:
        print(f"  {bm}: not enough non-zero tiers remain ({obs.shape[1]}) to run chi-square test — skipping.")
        continue

    chi2, p, dof, _ = stats.chi2_contingency(obs)
    print(f"  {bm}: chi2={chi2:.2f}  p={p:.2e}  dof={dof}"
          f"  {'← significantly different' if p < 0.05 else '(not significant)'}")
    print(f"       TopK tier shares: { {t: round(obs[0,i]/obs[0].sum(),3) for i,t in enumerate(kept_tiers)} }")
    print(f"       JFDS tier shares: { {t: round(obs[1,i]/obs[1].sum(),3) for i,t in enumerate(kept_tiers)} }")

---
## R10b — Bootstrap Confidence Interval for Aggregate Diversity

Aggregate diversity (distinct items covered across all users) is a
**catalogue-level** statistic, not a per-user one — there's no meaningful
"per-user aggregate diversity" to pair the way R7/R8 pair per-user metrics.
Instead we bootstrap-resample the user population with replacement, recompute
aggregate diversity from each resample's union of recommendations, and read
off a 95% CI and an empirical one-sided p-value for JFDS vs. each baseline.


In [ ]:
# ── Bootstrap CI: Aggregate Diversity, JFDS vs each baseline ─────────────
N_BOOT_AGGDIV = 200  # kept modest — resampling the full ~6,000-user population is not free

def build_incidence_matrix(rec_lists_dict, n_items=N_ITEMS):
    """Boolean (n_users, n_items) matrix: which items each user's list contains."""
    users_sorted = sorted(rec_lists_dict.keys())
    mat = np.zeros((len(users_sorted), n_items), dtype=bool)
    for i, u in enumerate(users_sorted):
        mat[i, rec_lists_dict[u]] = True
    return users_sorted, mat

def bootstrap_aggdiv_diff(mat_base, mat_jfds, n_boot=N_BOOT_AGGDIV, seed=RANDOM_SEED):
    """Resample users with replacement; return array of (AggDiv_JFDS - AggDiv_base) / N_ITEMS per resample."""
    rng = np.random.default_rng(seed)
    n = mat_base.shape[0]
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx_b = rng.integers(0, n, n)
        agg_base = mat_base[idx_b].any(axis=0).sum()
        agg_jfds = mat_jfds[idx_b].any(axis=0).sum()
        diffs[b] = (agg_jfds - agg_base) / N_ITEMS
    return diffs

aggdiv_boot_rows = []
for model_name in ['SVD', 'BPR']:
    users_jfds, mat_jfds = build_incidence_matrix(lists[(model_name, 'JFDS')])
    for baseline_method in BASELINE_METHODS:
        users_base, mat_base = build_incidence_matrix(lists[(model_name, baseline_method)])
        # Align on the common, identically-ordered user set before resampling jointly.
        common_users = sorted(set(users_jfds) & set(users_base))
        idx_j = [users_jfds.index(u) for u in common_users]
        idx_b = [users_base.index(u) for u in common_users]
        diffs = bootstrap_aggdiv_diff(mat_base[idx_b], mat_jfds[idx_j])

        ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
        p_emp = max((diffs <= 0).mean(), 1.0 / N_BOOT_AGGDIV)

        aggdiv_boot_rows.append({
            'base_model': model_name, 'baseline': baseline_method,
            'mean_diff (coverage frac)': diffs.mean(),
            '95% CI lower': ci_lo, '95% CI upper': ci_hi,
            'Empirical p (one-sided)': p_emp,
            'CI excludes 0': not (ci_lo <= 0 <= ci_hi),
        })

aggdiv_boot_df = pd.DataFrame(aggdiv_boot_rows)
print(f'=== Bootstrap comparison: Aggregate Diversity, JFDS vs each baseline ({N_BOOT_AGGDIV} resamples) ===')
print(aggdiv_boot_df.to_string(index=False, float_format=lambda x: f'{x:.4g}'))


---
## v3 Additions — Adaptive-λ JScore, Sensitivity, Overlap & Multi-Resample Robustness

Everything below ports over what `jfds_experiment_v3.ipynb` tested that this notebook
didn't: an **adaptive λ-schedule** (λ_f, λ_d change with list position instead of
staying fixed for the whole run), a **sensitivity sweep** across schedule shapes,
**item-overlap@K** between Top-K and the new adaptive equation, and a **robustness
check across many independent user resamples** with box plots and sign-stability.

These cells reuse the SVD/BPR models, candidate pools, and metric helpers already
built above — no retraining required.

### A. Adaptive-λ JFDS Equation

In every cell above, λ_f and λ_d were **constants for an entire run** (the Optuna
search found one best static pair per base model). Here they instead **grow with
list position** — early slots (small *k*) stay utility-heavy, later slots shift
weight toward fairness/diversity:

$$t_k = \frac{k}{K-1}, \qquad
\lambda_f(k) = \lambda_f^{start} + (\lambda_f^{end}-\lambda_f^{start})\cdot t_k^{\,p}, \qquad
\lambda_d(k) = \lambda_d^{start} + (\lambda_d^{end}-\lambda_d^{start})\cdot t_k^{\,p}$$

with $\lambda_u(k) = 1-\lambda_f(k)-\lambda_d(k)$ recomputed every step. $p=1$ is
linear, $p>1$ back-loaded, $p<1$ front-loaded. Reuses the same `greedy_rerank`
engine, `fair_boost`, and `diversity_term_fast` as fixed-λ JFDS above.

*(The equation above was already re-declared in the Setup cell -- `adaptive_jfds_rerank` is ready to use.)*

### B. Generate Adaptive-λ JFDS Recommendation Lists

In [ ]:
print('Re-ranking all users with Adaptive-λ JFDS ...')
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    adaptive_lists = {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        adaptive_lists[u] = adaptive_jfds_rerank(cand_ids, cand_rel)
    lists[(model_name, 'AdaptiveJFDS')] = adaptive_lists
    print(f"  {model_name}: {len(adaptive_lists)} Adaptive-JFDS lists  (schedule={DEFAULT_SCHEDULE})")

### C. Extend Evaluation Metrics to Cover Adaptive-λ JFDS

In [ ]:
adaptive_rows = []
for model_name in ['SVD', 'BPR']:
    rec_lists = lists[(model_name, 'AdaptiveJFDS')]
    for u, rec in rec_lists.items():
        relevant_set = test_relevant.get(u, set())
        grades       = test_grades.get(u, {})
        p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
        adaptive_rows.append({
            'user': u, 'base_model': model_name, 'method': 'AdaptiveJFDS',
            'precision': p, 'recall': r, 'ndcg': n_ndcg,
            'D': ild(rec), 'F': novelty_fairness(rec),
            'H': shannon_entropy(rec), 'gini_exp': gini(pop_count[rec]),
            'calibration_kl': calibration_kl(rec, train_seen.get(u, set())),
        })

adaptive_metrics_df = pd.DataFrame(adaptive_rows)
adaptive_metrics_df['U'] = adaptive_metrics_df['ndcg']
metrics_long = pd.concat([metrics_long, adaptive_metrics_df], ignore_index=True)
print(f"metrics_long now: {len(metrics_long):,} rows (added {len(adaptive_metrics_df):,} Adaptive-JFDS rows)")

# Headline comparison: Top-K vs fixed-λ JFDS vs Adaptive-λ JFDS
adaptive_summary = (metrics_long[metrics_long.method.isin(['TopK', 'JFDS', 'AdaptiveJFDS'])]
                     .groupby(['base_model', 'method'])[['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']]
                     .mean().round(4))
display(adaptive_summary)

### D. Item Overlap@K — Top-K vs Adaptive-λ JFDS

How many of the same movies does Adaptive-λ JFDS keep from the plain Top-K list,
per user? Low overlap means the equation is substituting, not just re-ordering.

In [ ]:
print('=== Item overlap@K: Top-K vs Adaptive-λ JFDS ===')
overlap_results = {}
for model_name in ['SVD', 'BPR']:
    topk_lists = lists[(model_name, 'TopK')]
    adap_lists = lists[(model_name, 'AdaptiveJFDS')]
    common_users = set(topk_lists) & set(adap_lists)
    overlaps = np.array([len(set(topk_lists[u]) & set(adap_lists[u])) for u in common_users])
    overlap_results[model_name] = overlaps
    print(f"  {model_name}: mean overlap@{K} = {overlaps.mean():.2f}  |  "
          f"median = {np.median(overlaps):.1f}  |  "
          f"users with 0 overlap = {(overlaps == 0).sum()} / {len(overlaps)} "
          f"({(overlaps == 0).mean() * 100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, model_name in zip(axes, ['SVD', 'BPR']):
    ax.hist(overlap_results[model_name], bins=np.arange(0, K + 2) - 0.5, color=C_JFDS, edgecolor='black', alpha=0.85)
    ax.set_xlabel(f'# items in common (Top-K ∩ Adaptive-JFDS), out of {K}')
    ax.set_ylabel('# users')
    ax.set_title(f'{model_name}: List overlap distribution')
    ax.set_xticks(range(0, K + 1))
plt.suptitle('Item Overlap@K — Top-K vs Adaptive-λ JFDS', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('overlap_topk_vs_adaptive.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 30-Resample Robustness Check

All results above (for Adaptive-λ JFDS) were computed on the full evaluated user population. To
check they aren't an artifact of a lucky user mix, redraw 30 independent random subsamples of
users (no retraining needed — reranking is already done for everyone) and see whether Adaptive-λ
JFDS's ordering relative to Top-K, MMR, and Fairness-only stays stable.

In [ ]:
N_REPEATS = 30
N_RESAMPLE = min(300, N_USERS)
RERUN_SEED_BASE = 1000
strategy_labels = ['TopK', 'MMR', 'FairOnly', 'AdaptiveJFDS']

rerun_rows = []
for model_name in ['SVD', 'BPR']:
    for rep in range(N_REPEATS):
        seed = RERUN_SEED_BASE + rep
        rng_rep = np.random.default_rng(seed)
        sampled_u = rng_rep.choice(N_USERS, size=N_RESAMPLE, replace=False)
        for method in strategy_labels:
            rec_lists = lists[(model_name, method)]
            all_sel = set()
            f_vals, d_vals, ndcg_vals = [], [], []
            for u in sampled_u:
                rec = rec_lists.get(u)
                if not rec:
                    continue
                all_sel.update(rec)
                f_vals.append(novelty_fairness(rec))
                d_vals.append(ild(rec))
                relevant_set = test_relevant.get(u, set())
                grades = test_grades.get(u, {})
                _, _, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
                ndcg_vals.append(n_ndcg)
            rerun_rows.append({
                'base_model': model_name, 'rep': rep, 'seed': seed, 'Strategy': method,
                'Fairness': np.mean(f_vals), 'AggDiv': len(all_sel) / N_ITEMS,
                'ILD': np.mean(d_vals), 'NDCG': np.mean(ndcg_vals),
            })

rerun_df = pd.DataFrame(rerun_rows)
print(f"Done — {len(rerun_df)} rows ({N_REPEATS} reruns × {len(strategy_labels)} strategies × 2 base models).")

rerun_summary = rerun_df.groupby(['base_model', 'Strategy'])[['Fairness', 'AggDiv', 'ILD', 'NDCG']].agg(['mean', 'std'])
display(rerun_summary.round(4))

In [ ]:
metrics_box = ['Fairness', 'AggDiv', 'ILD', 'NDCG']
colors_map = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22', 'AdaptiveJFDS': '#27AE60'}

for model_name in ['SVD', 'BPR']:
    sub_model = rerun_df[rerun_df.base_model == model_name]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, metric in zip(axes, metrics_box):
        data = [sub_model.loc[sub_model.Strategy == s, metric].values for s in strategy_labels]
        bp = ax.boxplot(data, patch_artist=True, widths=0.6, medianprops=dict(color='black', linewidth=1.5))
        for patch, s in zip(bp['boxes'], strategy_labels):
            patch.set_facecolor(colors_map[s]); patch.set_alpha(0.75)
        rng_jit = np.random.default_rng(0)
        for i, s in enumerate(strategy_labels):
            vals = sub_model.loc[sub_model.Strategy == s, metric].values
            jitter = rng_jit.uniform(-0.12, 0.12, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals, color='black', s=8, alpha=0.4, zorder=3)
        ax.set_title(metric, fontsize=11)
        ax.set_xticks(range(1, len(strategy_labels) + 1))
        ax.set_xticklabels(strategy_labels, rotation=20, ha='right')
    plt.suptitle(f'{model_name}: Distribution across {N_REPEATS} independent resamples of {N_RESAMPLE} users each',
                 y=1.08, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'resample_boxplots_{model_name.lower()}.png', dpi=110, bbox_inches='tight')
    plt.show()

In [ ]:
baseline_order = ['TopK', 'MMR', 'FairOnly']
baseline_colors = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22'}
sign_stability_rows = []

for model_name in ['SVD', 'BPR']:
    sub_model = rerun_df[rerun_df.base_model == model_name]
    wide = sub_model.pivot(index='rep', columns='Strategy', values=metrics_box)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, metric in zip(axes, metrics_box):
        adap_vals = wide[metric]['AdaptiveJFDS'].values
        deltas = {b: adap_vals - wide[metric][b].values for b in baseline_order}
        bp = ax.boxplot([deltas[b] for b in baseline_order], patch_artist=True, widths=0.6,
                         medianprops=dict(color='black', linewidth=1.5))
        for patch, b in zip(bp['boxes'], baseline_order):
            patch.set_facecolor(baseline_colors[b]); patch.set_alpha(0.75)
        ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title(f'AdaptiveJFDS − baseline\n{metric}', fontsize=10)
        ax.set_xticks(range(1, len(baseline_order) + 1))
        ax.set_xticklabels(baseline_order, rotation=20, ha='right')
    plt.suptitle(f'{model_name}: Sign stability across {N_REPEATS} resamples (AdaptiveJFDS vs baselines)',
                 y=1.1, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'sign_stability_{model_name.lower()}.png', dpi=110, bbox_inches='tight')
    plt.show()

    print(f'=== {model_name}: fraction of {N_REPEATS} reruns where AdaptiveJFDS beats each baseline ===')
    for metric in metrics_box:
        adap_vals = wide[metric]['AdaptiveJFDS'].values
        for b in baseline_order:
            base_vals = wide[metric][b].values
            frac = (adap_vals > base_vals).mean()
            print(f'  {metric:10s} vs {b:10s}: AdaptiveJFDS higher in {frac * 100:5.1f}% of reruns')
            sign_stability_rows.append({'base_model': model_name, 'metric': metric, 'baseline': b, 'frac_adaptive_wins': frac})
    print()

sign_stability_df = pd.DataFrame(sign_stability_rows)
display(sign_stability_df)

### Summary of v3-Ported Sections

- **Adaptive-λ JFDS** (`AdaptiveJFDS`) is now a fifth strategy alongside Top-K,
  JFDS, MMR, and Fairness-only, generated for the **full** user population on
  both SVD and BPR — see `adaptive_summary` for the headline comparison.
- **`overlap_results`** holds per-user item-overlap@K counts (Top-K vs
  Adaptive-JFDS) for both base models.
- **`sens_df`** holds the schedule-shape sensitivity sweep (linear / front-loaded /
  back-loaded / wider-range) with Fairness, AggDiv, ILD, JFDS-composite, and NDCG.
- **`rerun_df`** / **`sign_stability_df`** hold the 30-resample robustness check —
  box plots show whether Adaptive-JFDS's ranking relative to each baseline is
  stable across independently drawn user subsamples, and the win-fraction table
  quantifies how often the sign ever flips.

---
## Conclusion

In [ ]:
n_sig_favor_jfds = int(
    (((sig_table['wilcoxon_p_holm'] < 0.05) & (sig_table['mean_delta'] > 0) & (sig_table['metric'] != 'calibration_kl')).sum()) +
    (((sig_table['wilcoxon_p_holm'] < 0.05) & (sig_table['mean_delta'] < 0) & (sig_table['metric'] == 'calibration_kl')).sum())
)
n_metrics_tested = len(sig_table)
n_aggdiv_sig = int(aggdiv_boot_df['CI excludes 0'].sum())

display(Markdown(f'''
### Hypothesis-Test Summary

Out of **{n_metrics_tested}** (metric x baseline x base-model) comparisons in R7 -- covering
**Top-K, MMR, and Fairness-only** as baselines -- JFDS is significantly better
(Wilcoxon p<0.05, **Holm-Bonferroni corrected** for all {n_metrics_tested} comparisons) on
**{n_sig_favor_jfds}** of them. See `sig_table` for the full per-comparison breakdown, including
which specific baseline/metric/model combinations do *not* favor JFDS -- JFDS is not expected to
beat MMR on diversity or Fairness-only on raw fairness, since each specialist baseline should be
strongest on the one thing it optimizes alone. The interesting claim (H1 from
`01_theory_and_hypothesis.ipynb`) is that JFDS wins on the **joint** objective without being
catastrophically behind either specialist on its own metric.

**Baseline comparison:** see `baseline_delta_df` (persisted from `02_main_experiments`) for
exactly how much JFDS gains or gives up vs. Top-K, MMR, and Fairness-only on Fairness / Diversity / NDCG.

**R7/R7b/R8:** `sig_table` (Holm-corrected p-values) and `boot_df` hold exact p-values and effect
sizes per metric, per baseline, per model.

**R9 (Group fairness):** see the bar charts and Kruskal-Wallis results above -- check whether
JFDS's gains are uniform across gender, age, and occupation groups, or concentrated in some.

**R10/R10b (Aggregate diversity):** see `agg_df` for distinct-item counts and Lorenz Gini per
strategy, chi-square results for tier-exposure shifts, and `aggdiv_boot_df` for bootstrap
confidence intervals confirming (or not) that JFDS's aggregate-diversity advantage over each
baseline holds up beyond a single point estimate -- **{n_aggdiv_sig} / {len(aggdiv_boot_df)}**
baseline comparisons have a 95% CI that excludes zero.

**Adaptive-λ JFDS (H3):** letting λ_f and λ_d grow with list position (instead of staying fixed
for the whole run) gives `AdaptiveJFDS`, evaluated on the full user population for both SVD and
BPR -- see `adaptive_summary` for headline numbers vs. Top-K and fixed-λ JFDS. Item-overlap@K
(`overlap_results`) shows how much the adaptive list actually substitutes rather than just
re-orders Top-K. The 30-independent-resample robustness check (`rerun_df`, `sign_stability_df`)
confirms whether Adaptive-JFDS's ranking relative to Top-K / MMR / Fairness-only is a stable
property of the dataset or an artifact of one particular user sample.

**Schedule-shape sensitivity and the F-D trade-off surface (R1-R6):** see
`05_appendix_supplementary.ipynb` for the exploratory analyses that motivated some of the design
choices above but are not part of the confirmatory claim.
'''))


### Limitations

- **Single dataset.** All results are on MovieLens 1M, a research-grade, relatively clean,
  explicit-feedback dataset. Popularity-tier structure, genre coverage, and user-behaviour
  patterns differ substantially in production-scale implicit-feedback catalogues (e-commerce,
  short video, music streaming); the fairness/diversity gains observed here may not transfer
  directly in magnitude, even if the qualitative ranking (JFDS ≥ single-objective baselines on
  the joint criterion) is likely to hold given the theoretical argument in `01`.
- **Linear scalarization only.** JFDS combines relevance, fairness, and diversity as a convex
  (linear) combination. This can only ever reach the *concave* part of the Pareto frontier
  (see `01_theory_and_hypothesis.ipynb`, "Scalarization" section) — if the true frontier has a
  non-convex region, JFDS cannot represent solutions there regardless of λ. R1 in the appendix
  gives a first empirical look at how much of the frontier is actually reachable.
- **Static popularity tiers.** Head/mid/tail tiers are computed once from the training split and
  held fixed. In a live system, item popularity shifts continuously; a deployed JFDS would need
  either periodic tier recomputation or an online/streaming variant of `fair_boost`.
- **Genre-based diversity/similarity only.** `GENRE_SIM` uses multi-hot genre cosine similarity as
  the sole notion of "similar item." This is a coarse proxy — two action movies from different
  decades and audiences look identical to this metric. Content- or embedding-based similarity
  (e.g. from a trained item encoder) would likely change both the diversity term's behaviour and
  the reported ILD numbers.
- **Greedy, not globally optimal, re-ranking.** `greedy_rerank` makes a locally optimal choice at
  each of the k positions; it is not guaranteed to find the k-subset that globally maximizes the
  JFDS objective. This is standard practice in the re-ranking literature (MMR itself is greedy),
  but it does mean the reported numbers are a lower bound on what an exact (or better-approximated)
  solver could achieve.
- **Ablation and complexity results in `03` were computed on the same dataset/split**, so
  variance estimates there share the same single-dataset caveat as above.

### Practical Implications

- **JFDS is a drop-in re-ranking layer**, not a replacement recommender — it operates purely on
  top of any base scorer's candidate pool (demonstrated here on both SVD and BPR), so it can sit
  behind an existing production ranking model without retraining it.
- **The gap-squared fairness term is self-limiting**: because the penalty is `(gap/target)²`, its
  contribution vanishes smoothly as a tier approaches its target share (see `01` for the
  derivation), which avoids the oscillation/overshoot failure mode a naive proportional (linear
  gap) controller can exhibit — relevant for systems where fairness constraints must be enforced
  continuously online, not just at evaluation time.
- **The adaptive-λ schedule is the more deployment-relevant variant** if position-bias in
  click-through matters to the product: it concentrates the fairness/diversity cost at the ranks
  where a user is least likely to notice a lower-relevance item (see the DCG-discount argument in
  `01`), which is a cheap way to reduce the NDCG cost of a fairness intervention without any change
  to the loss weights `λ_f`, `λ_d` themselves.
- **Runtime is the main deployment constraint**, not accuracy: `03_ablation_and_complexity` shows
  JFDS/MMR/AdaptiveJFDS scale as O(P·k²) in candidate-pool size P and list length k, versus O(P·k)
  for Fairness-only and O(P log P) for Top-K — for latency-sensitive serving, capping the candidate
  pool size P is the first lever to pull, not shortening the list k.

### Future Work

- **Learned λ, not searched λ.** The current pipeline picks one (or a scheduled) λ_f, λ_d pair via
  Optuna against a fixed validation objective. A natural extension is a contextual/per-user λ,
  learned end-to-end, so that fairness/diversity pressure adapts to each user's own tolerance for
  novel or unfamiliar recommendations rather than using one global setting.
- **Beyond genre similarity.** Replacing `GENRE_SIM` with a learned item-embedding similarity
  (e.g. from the SVD/BPR item factors themselves, or a separate content encoder) would let the
  diversity term capture similarity the way users actually perceive it, not just shared genre tags.
- **Exact or better-approximated re-ranking.** Comparing greedy JFDS against an integer-programming
  or beam-search re-ranker would quantify how much objective value is lost to the greedy
  approximation, and whether that gap is worth the runtime cost in production settings with looser
  latency budgets.
- **Cross-dataset replication.** Repeating the full pipeline (04's significance tests in
  particular) on a second, structurally different dataset — implicit feedback, larger catalogue,
  different popularity skew — would directly test the single-dataset limitation above.
- **Online/bandit evaluation.** All results here are offline, computed against a held-out test
  split. A/B or bandit-based online evaluation would be needed to confirm that fairness/diversity
  gains measured offline translate into any change in actual user engagement or satisfaction.